# QnA Datasets Combined

Load all QnA datasets and show basic statistics.

**Single-turn** (context → question → answer):
- SQuAD v2 (has unanswerable)
- Natural Questions (has unanswerable)
- TriviaQA
- NewsQA
- MRQA (SearchQA + HotpotQA subsets only)
- AdversarialQA
- SQuAD v1 (dedup against v2)

**Multi-turn / Conversational** (has unanswerable):
- QuAC
- CoQA

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path

import pandas as pd
from datasets import load_dataset

DATA_PATH = Path('Q:/data')
QNA_DATA_PATH = DATA_PATH / 'qna'
QNA_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f'DATA_PATH: {DATA_PATH}')
print(f'QNA_DATA_PATH: {QNA_DATA_PATH}')

DATA_PATH: Q:\data
QNA_DATA_PATH: Q:\data\qna


## Single-turn datasets

In [14]:
# --- SQuAD v2 ---
ds_squad_v2 = load_dataset('rajpurkar/squad_v2', cache_dir=str(DATA_PATH))
print('SQuAD v2:', ds_squad_v2)

# --- Natural Questions ---
ds_nq_train = load_dataset('google-research-datasets/natural_questions', split='train', cache_dir=str(DATA_PATH))
ds_nq_val = load_dataset('google-research-datasets/natural_questions', split='validation', cache_dir=str(DATA_PATH))
print(f'Natural Questions: train={len(ds_nq_train)}, val={len(ds_nq_val)}')

# --- TriviaQA ---
ds_triviaqa = load_dataset('trivia_qa', 'rc', cache_dir=str(DATA_PATH))
print('TriviaQA:', ds_triviaqa)

# --- NewsQA ---
ds_newsqa = load_dataset('lucadiliello/newsqa', cache_dir=str(DATA_PATH))
print('NewsQA:', ds_newsqa)

# --- MRQA ---
ds_mrqa = load_dataset('mrqa', cache_dir=str(DATA_PATH))
print('MRQA:', ds_mrqa)

# --- AdversarialQA ---
ds_adversarialqa = load_dataset('adversarial_qa', 'adversarialQA', cache_dir=str(DATA_PATH))
print('AdversarialQA:', ds_adversarialqa)

# --- SQuAD v1 ---
ds_squad_v1 = load_dataset('rajpurkar/squad', cache_dir=str(DATA_PATH))
print('SQuAD v1:', ds_squad_v1)

SQuAD v2: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/235 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

...

Natural Questions: train=307373, val=7830


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

TriviaQA: DatasetDict({
    train: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 138384
    })
    validation: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17944
    })
    test: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17210
    })
})
NewsQA: DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answers', 'key', 'labels'],
        num_rows: 74160
    })
    validation: Dataset({
        features: ['context', 'question', 'answers', 'key', 'labels'],
        num_rows: 4212
    })
})
MRQA: DatasetDict({
    train: Dataset({
        features: ['subset', 'context', 'context_tokens', 'qid', 'question', 'question_tokens', 'detected_answers', 'answers'],
        num_rows: 516819
    })
    va

## Multi-turn / Conversational datasets

In [15]:
# --- QuAC ---
# QuAC still has a legacy loading script (quac.py) which is no longer supported.
# Use HF's auto-converted Parquet export instead.
ds_quac = load_dataset('quac', revision='refs/convert/parquet', cache_dir=str(DATA_PATH))
print('QuAC:', ds_quac)

# --- CoQA ---
ds_coqa = load_dataset('stanfordnlp/coqa', cache_dir=str(DATA_PATH))
print('CoQA:', ds_coqa)

QuAC: DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'wikipedia_page_title', 'background', 'section_title', 'context', 'turn_ids', 'questions', 'followups', 'yesnos', 'answers', 'orig_answers'],
        num_rows: 11567
    })
    validation: Dataset({
        features: ['dialogue_id', 'wikipedia_page_title', 'background', 'section_title', 'context', 'turn_ids', 'questions', 'followups', 'yesnos', 'answers', 'orig_answers'],
        num_rows: 1000
    })
})
CoQA: DatasetDict({
    train: Dataset({
        features: ['source', 'story', 'questions', 'answers'],
        num_rows: 7199
    })
    validation: Dataset({
        features: ['source', 'story', 'questions', 'answers'],
        num_rows: 500
    })
})


## Basic statistics

In [16]:
def count_unanswerable_squad(split_ds):
    """SQuAD v2 format: unanswerable when answers['text'] is empty."""
    return sum(1 for ex in split_ds if len(ex['answers']['text']) == 0)

def count_unanswerable_quac(split_ds):
    """QuAC: unanswerable turns have answer == 'CANNOTANSWER'."""
    total_turns = sum(len(ex['questions']) for ex in split_ds)
    unans_turns = sum(
        sum(1 for a in ex['orig_answers']['texts'] if a == 'CANNOTANSWER')
        for ex in split_ds
    )
    return total_turns, unans_turns

def count_unanswerable_coqa(split_ds):
    """CoQA: unanswerable turns have input_text == 'unknown'."""
    total_turns = sum(len(ex['questions']) for ex in split_ds)
    unans_turns = sum(
        sum(1 for a in ex['answers']['input_text'] if a.lower() == 'unknown')
        for ex in split_ds
    )
    return total_turns, unans_turns

def count_nq_unanswerable(split_ds):
    """NQ: unanswerable when no short answer exists."""
    n_unans = 0
    for ex in split_ds:
        has_short = any(
            len(sa['start_token']) > 0
            for sa in ex['annotations']['short_answers']
        )
        if not has_short:
            n_unans += 1
    return n_unans

def mrqa_subset_counts(split_ds):
    """Count rows per MRQA subset."""
    from collections import Counter
    return Counter(ex['subset'] for ex in split_ds)


# ---- Collect stats ----
stats = []

# SQuAD v2
for split_name in ['train', 'validation']:
    ds = ds_squad_v2[split_name]
    n_unans = count_unanswerable_squad(ds)
    stats.append({
        'dataset': 'SQuAD v2', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': n_unans,
        'unanswerable_pct': f'{n_unans / len(ds) * 100:.1f}%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# Natural Questions
for split_name, ds in [('train', ds_nq_train), ('validation', ds_nq_val)]:
    n_unans = count_nq_unanswerable(ds)
    stats.append({
        'dataset': 'Natural Questions', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': n_unans,
        'unanswerable_pct': f'{n_unans / len(ds) * 100:.1f}%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# TriviaQA
for split_name in ds_triviaqa:
    ds = ds_triviaqa[split_name]
    stats.append({
        'dataset': 'TriviaQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# NewsQA
for split_name in ds_newsqa:
    ds = ds_newsqa[split_name]
    stats.append({
        'dataset': 'NewsQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# MRQA (split by subset)
for split_name in ds_mrqa:
    ds = ds_mrqa[split_name]
    sub_counts = mrqa_subset_counts(ds)
    for sub_name, sub_count in sorted(sub_counts.items()):
        stats.append({
            'dataset': f'MRQA/{sub_name}', 'split': split_name, 'rows': sub_count,
            'total_qa_pairs': sub_count, 'unanswerable': 0,
            'unanswerable_pct': '0.0%',
            'multi_turn': False, 'columns': list(ds.column_names),
        })

# AdversarialQA
for split_name in ds_adversarialqa:
    ds = ds_adversarialqa[split_name]
    stats.append({
        'dataset': 'AdversarialQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# SQuAD v1
for split_name in ds_squad_v1:
    ds = ds_squad_v1[split_name]
    stats.append({
        'dataset': 'SQuAD v1', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# QuAC (multi-turn)
for split_name in ds_quac:
    ds = ds_quac[split_name]
    total_turns, unans_turns = count_unanswerable_quac(ds)
    stats.append({
        'dataset': 'QuAC', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': total_turns, 'unanswerable': unans_turns,
        'unanswerable_pct': f'{unans_turns / total_turns * 100:.1f}%',
        'multi_turn': True, 'columns': list(ds.column_names),
    })

# CoQA (multi-turn)
for split_name in ds_coqa:
    ds = ds_coqa[split_name]
    total_turns, unans_turns = count_unanswerable_coqa(ds)
    stats.append({
        'dataset': 'CoQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': total_turns, 'unanswerable': unans_turns,
        'unanswerable_pct': f'{unans_turns / total_turns * 100:.1f}%',
        'multi_turn': True, 'columns': list(ds.column_names),
    })

df_stats = pd.DataFrame(stats)
print(f'Total datasets entries: {len(df_stats)}')
print()

# Summary table (without columns list for readability)
display_cols = ['dataset', 'split', 'rows', 'total_qa_pairs', 'unanswerable', 'unanswerable_pct', 'multi_turn']
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 40):
    display(df_stats[display_cols])

Total datasets entries: 36



,dataset,split,rows,total_qa_pairs,unanswerable,unanswerable_pct,multi_turn
0,SQuAD v2,train,130319,130319,43498,33.4%,False
1,SQuAD v2,validation,11873,11873,5945,50.1%,False
2,Natural Questions,train,307373,307373,200447,65.2%,False
3,Natural Questions,validation,7830,7830,3541,45.2%,False
4,TriviaQA,train,138384,138384,0,0.0%,False
5,TriviaQA,validation,17944,17944,0,0.0%,False
6,TriviaQA,test,17210,17210,0,0.0%,False
7,NewsQA,train,74160,74160,0,0.0%,False
8,NewsQA,validation,4212,4212,0,0.0%,False
9,MRQA/HotpotQA,train,72928,72928,0,0.0%,False
